# 01. ETL: Профиль клиента / проспекта

> **Краткое резюме**: Извлекаем профильные данные для каждого клиента/контрагента банка: тип, статус (клиент/контрагент), ОКВЭД, регион, модельный сегмент, уставный капитал. Результат — `data/client_profiles.parquet`.

**Источники данных**:
- `client_sdim` + `clienttype_ldim` — базовые атрибуты
- `clientinfo_shist` — флаг клиент/контрагент, дата перехода
- `dmoutdmrb_mdm_redclients_shist_dmout` — ОКВЭД (СПАРК), город
- `client2smartsplace_shist` — регион, город
- `dmrb_clientgeneralsegment_shist` — модельный сегмент
- `sparkfinindicator_sstat` — выручка (опционально)

**Предпосылки**: Доступ к Hive-базе `s_dmrb`.

---

In [ ]:
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
log = logging.getLogger(__name__)

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

HIVE_DATABASE = 's_dmrb'

# Период анализа (6 месяцев — баланс между объёмом и репрезентативностью)
START_DATE = '2025-07-01'
END_DATE   = '2025-12-31'

# as_of_date для SCD-таблиц (effective_to >= as_of_date)
AS_OF_DATE = END_DATE

# Целочисленные даты для partitioned таблиц (date_part INT YYYYMMDD)
START_INT = int(START_DATE.replace('-', ''))
END_INT   = int(END_DATE.replace('-', ''))

# Выходная директория
OUTPUT_DIR = os.path.abspath('./data')
os.makedirs(OUTPUT_DIR, exist_ok=True)
SPARK_OUTPUT_DIR = f'file://{OUTPUT_DIR}'

print(f'Period: {START_DATE} — {END_DATE} (date_part {START_INT}..{END_INT})')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
try:
    _ = spark.sparkContext.appName
    print(f'SparkSession available: {spark.sparkContext.appName}')
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName('LookAlike_ETL_Profile') \
        .enableHiveSupport() \
        .getOrCreate()
    print('SparkSession created')

spark.sql(f'USE {HIVE_DATABASE}')
print(f'Spark {spark.version}, database: {HIVE_DATABASE}')

---
## 1. Вселенная клиентов

Все `client_uk`, которые участвовали в транзакциях за период (как плательщик или получатель). Это наш рабочий набор — активные сущности банка.

In [ ]:
universe_df = spark.sql(f"""
    SELECT DISTINCT client_uk
    FROM {HIVE_DATABASE}.paymentcounteragent_stran
    WHERE date_part >= {START_INT} AND date_part <= {END_INT}
      AND (deleted_flag IS NULL OR deleted_flag != 'Y')
      AND client_uk IS NOT NULL
""").cache()

universe_count = universe_df.count()
print(f'Universe: {universe_count:,} unique client_uk')

# Сохраняем в Parquet и перечитываем как temp view
# (надёжнее чем .cache() для JOIN с большими таблицами)
universe_parquet = f'{SPARK_OUTPUT_DIR}/universe.parquet'
universe_df.write.mode('overwrite').parquet(universe_parquet)
spark.read.parquet(universe_parquet).createOrReplaceTempView('universe')
print('Universe saved to Parquet and registered as temp view')

---
## 2. Базовые атрибуты (`client_sdim`)

In [ ]:
base_df = spark.sql(f"""
    SELECT
        u.client_uk,
        cl.client_name,
        cl.client_pin,
        cl.entrepreneur_flag,
        cl.blacklist_flag,
        ct.name AS client_type_name
    FROM universe u
    INNER JOIN {HIVE_DATABASE}.client_sdim cl
        ON u.client_uk = cl.uk
    LEFT JOIN {HIVE_DATABASE}.clienttype_ldim ct
        ON cl.clienttype_uk = ct.uk
""")

# Сохраняем pin-маппинг для segment-запроса (cell-17)
pin_map_path = f'{SPARK_OUTPUT_DIR}/pin_map.parquet'
base_df.select('client_uk', 'client_pin').where('client_pin IS NOT NULL') \
    .write.mode('overwrite').parquet(pin_map_path)
spark.read.parquet(pin_map_path).createOrReplaceTempView('pin_map')
print(f'Pin map saved and registered as temp view')

base_df.createOrReplaceTempView('base_clients')
print(f'Base attributes: {base_df.count():,} rows')
base_df.show(5, truncate=40)

---
## 3. Статус клиент / контрагент (`clientinfo_shist`)

- `clientcounterparty_flag`: **Y** = клиент, **N** = контрагент (проспект), **?** = неизвестно
- `clientchange_date` — дата перехода в клиенты (если был контрагентом)
- `client_rel_date` — дата начала отношений

In [ ]:
info_df = spark.sql(f"""
    SELECT client_uk, clientcounterparty_flag, clientchange_date,
           counterpartychange_date, client_rel_date,
           registry_authcapital_amt, paid_authcapital_amt,
           loanapplscope_uk
    FROM (
        SELECT ci.*,
               ROW_NUMBER() OVER (PARTITION BY ci.client_uk
                                  ORDER BY ci.effective_from DESC) AS rn
        FROM {HIVE_DATABASE}.clientinfo_shist ci
        INNER JOIN universe u ON ci.client_uk = u.client_uk
        WHERE (ci.effective_to >= '{AS_OF_DATE}' OR ci.effective_to IS NULL)
          AND (ci.deleted_flag IS NULL OR ci.deleted_flag != 'Y')
    ) t WHERE rn = 1
""")

info_df.createOrReplaceTempView('client_info')
print(f'Client info: {info_df.count():,} rows')

# Распределение по флагу
info_df.groupBy('clientcounterparty_flag').count().show()

---
## 4. ОКВЭД и город (`dmoutdmrb_mdm_redclients_shist_dmout`)

Таблица связывает ЮЛ (`client_lp_uk`) с ФЛ (`client_np_uk`).  
Извлекаем атрибуты ЮЛ: ОКВЭД из СПАРК, город регистрации, статус.

In [ ]:
okved_df = spark.sql(f"""
    SELECT client_uk, sparkokved_ccode, reg_city_name, statusfu_ccode
    FROM (
        SELECT
            r.client_lp_uk AS client_uk,
            r.sparkokved_ccode,
            r.reg_city_name,
            r.statusfu_ccode,
            ROW_NUMBER() OVER (PARTITION BY r.client_lp_uk
                               ORDER BY r.effective_from DESC) AS rn
        FROM {HIVE_DATABASE}.dmoutdmrb_mdm_redclients_shist_dmout r
        INNER JOIN universe u ON r.client_lp_uk = u.client_uk
        WHERE r.client_lp_uk IS NOT NULL
          AND (r.effective_to >= '{AS_OF_DATE}' OR r.effective_to IS NULL)
          AND (r.deleted_flag IS NULL OR r.deleted_flag != 'Y')
    ) t WHERE rn = 1
""")

print(f'OKVED data: {okved_df.count():,} rows')
okved_df.show(5, truncate=30)

---
## 5. Регион (`client2smartsplace_shist`)

In [ ]:
region_df = spark.sql(f"""
    SELECT client_uk, addrref_region_uk, addrref_city_name
    FROM (
        SELECT s.client_uk, s.addrref_region_uk, s.addrref_city_name,
               ROW_NUMBER() OVER (PARTITION BY s.client_uk
                                  ORDER BY s.effective_from DESC) AS rn
        FROM {HIVE_DATABASE}.client2smartsplace_shist s
        INNER JOIN universe u ON s.client_uk = u.client_uk
        WHERE s.client_uk IS NOT NULL
          AND (s.effective_to >= '{AS_OF_DATE}' OR s.effective_to IS NULL)
          AND (s.deleted_flag IS NULL OR s.deleted_flag != 'Y')
    ) t WHERE rn = 1
""")

print(f'Region data: {region_df.count():,} rows')
region_df.show(5, truncate=30)

---
## 6. Модельный сегмент (`dmrb_clientgeneralsegment_shist`)

In [ ]:
# Диагностика: почему 0 строк?
print('=== 1. Всего строк в таблице ===')
spark.sql(f'SELECT COUNT(*) AS total FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist').show()

print('=== 2. Примеры effective_from / effective_to ===')
spark.sql(f"""
    SELECT effective_from, effective_to, deleted_flag, client_uk, client_pin,
           srvpackagesegment_ccode
    FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist
    LIMIT 10
""").show(truncate=False)

print('=== 3. Распределение effective_to ===')
spark.sql(f"""
    SELECT
        CASE
            WHEN effective_to IS NULL THEN 'NULL'
            WHEN effective_to >= '2025-12-31' THEN '>= 2025-12-31'
            WHEN effective_to >= '2025-01-01' THEN '2025-xx-xx'
            WHEN effective_to >= '2020-01-01' THEN '2020-2024'
            ELSE '< 2020'
        END AS eff_to_bucket,
        COUNT(*) AS cnt
    FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist
    GROUP BY 1
    ORDER BY cnt DESC
""").show(truncate=False)

print('=== 4. Сколько строк матчится с universe (без SCD-фильтра) ===')
spark.sql(f"""
    SELECT COUNT(*) AS matched
    FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist g
    INNER JOIN universe u ON g.client_uk = u.client_uk
""").show()

print('=== 5. Сколько строк матчится по client_pin (если client_uk пустой) ===')
spark.sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(client_uk) AS has_uk,
        COUNT(client_pin) AS has_pin
    FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist
    LIMIT 1
""").show()

In [ ]:
# pin_map уже создан в cell-7 (client_uk → client_pin из client_sdim)
print('pin_map temp view:')
spark.sql('SELECT COUNT(*) AS cnt FROM pin_map').show()

In [ ]:
# Шаг 2: Segment через pin_map (уже на диске, лёгкий JOIN)

segment_df = spark.sql(f"""
    SELECT t.client_uk,
           t.srvpackagesegment_ccode, t.srvpackagelightsegment_ccode,
           t.agesegment_ccode, t.client_active_flag
    FROM (
        SELECT pm.uni_client_uk AS client_uk,
               g.srvpackagesegment_ccode, g.srvpackagelightsegment_ccode,
               g.agesegment_ccode, g.client_active_flag,
               ROW_NUMBER() OVER (PARTITION BY pm.uni_client_uk
                                  ORDER BY g.effective_from DESC) AS rn
        FROM {HIVE_DATABASE}.dmrb_clientgeneralsegment_shist g
        INNER JOIN pin_map pm ON g.client_pin = pm.cl_pin
        WHERE g.client_pin IS NOT NULL
          AND (g.effective_to >= '{AS_OF_DATE}' OR g.effective_to IS NULL)
          AND (g.deleted_flag IS NULL OR g.deleted_flag != 'Y')
    ) t
    WHERE t.rn = 1
""")

seg_count = segment_df.count()
print(f'Segment data: {seg_count:,} rows')

if seg_count > 0:
    segment_df.groupBy('srvpackagesegment_ccode').count() \
        .orderBy('count', ascending=False).show(20, truncate=False)
else:
    print('WARNING: No segment data. Will be NULL in profile.')

---
## 7. Выручка из СПАРК (опционально)

Связь через `sparkcompany_src_ccode` (string) → `sparkcompany_uk` (double).  
Прямого FK нет — пробуем CAST. Если не работает, пропускаем.

In [ ]:
try:
    revenue_df = spark.sql(f"""
        SELECT
            r.client_lp_uk AS client_uk,
            f.revenue_rur_amt,
            f.adjrevenue_rur_amt,
            f.failurescore_ccode,
            f.duediligense_ccode,
            f.paymentindex_ccode
        FROM (
            SELECT client_lp_uk, sparkcompany_src_ccode,
                   ROW_NUMBER() OVER (PARTITION BY client_lp_uk
                                      ORDER BY effective_from DESC) AS rn
            FROM {HIVE_DATABASE}.dmoutdmrb_mdm_redclients_shist_dmout r2
            INNER JOIN universe u ON r2.client_lp_uk = u.client_uk
            WHERE r2.client_lp_uk IS NOT NULL
              AND r2.sparkcompany_src_ccode IS NOT NULL
              AND r2.sparkcompany_src_ccode != ''
              AND (r2.effective_to >= '{AS_OF_DATE}' OR r2.effective_to IS NULL)
              AND (r2.deleted_flag IS NULL OR r2.deleted_flag != 'Y')
        ) r
        JOIN (
            SELECT sparkcompany_uk, revenue_rur_amt, adjrevenue_rur_amt,
                   failurescore_ccode, duediligense_ccode, paymentindex_ccode,
                   ROW_NUMBER() OVER (PARTITION BY sparkcompany_uk
                                      ORDER BY value_day DESC) AS rn2
            FROM {HIVE_DATABASE}.sparkfinindicator_sstat
            WHERE revenue_rur_amt IS NOT NULL
              AND (deleted_flag IS NULL OR deleted_flag != 'Y')
        ) f
            ON CAST(TRIM(r.sparkcompany_src_ccode) AS DOUBLE) = f.sparkcompany_uk
            AND f.rn2 = 1
        WHERE r.rn = 1
    """)

    rev_count = revenue_df.count()
    print(f'SPARK revenue: {rev_count:,} rows matched')
    if rev_count > 0:
        revenue_df.describe('revenue_rur_amt').show()
    else:
        print('No matches — CAST join failed. Revenue will be NULL.')
        revenue_df = None

except Exception as e:
    print(f'SPARK revenue extraction failed: {e}')
    print('Skipping — bank turnover from 02_etl_behavior will be used as proxy.')
    revenue_df = None

---
## 8. Сборка профиля

LEFT JOIN всех источников по `client_uk`.

In [ ]:
profile_df = base_df \
    .join(info_df, on='client_uk', how='left') \
    .join(okved_df, on='client_uk', how='left') \
    .join(region_df, on='client_uk', how='left') \
    .join(segment_df, on='client_uk', how='left')

if revenue_df is not None:
    profile_df = profile_df.join(revenue_df, on='client_uk', how='left')

print(f'Profile: {profile_df.count():,} rows, {len(profile_df.columns)} columns')
print(f'Columns: {profile_df.columns}')

In [ ]:
# Сохранение
out_path = f'{SPARK_OUTPUT_DIR}/client_profiles.parquet'
profile_df.write.mode('overwrite').parquet(out_path)
print(f'Saved: {out_path}')

---
## 9. Сводная статистика

In [ ]:
print('=== Распределение по типу клиента ===')
profile_df.groupBy('client_type_name').count().orderBy('count', ascending=False).show()

print('=== Распределение по флагу клиент/контрагент ===')
profile_df.groupBy('clientcounterparty_flag').count().orderBy('count', ascending=False).show()

print('=== Топ-15 ОКВЭД ===')
profile_df.filter('sparkokved_ccode IS NOT NULL') \
    .groupBy('sparkokved_ccode').count() \
    .orderBy('count', ascending=False).show(15, truncate=False)

print('=== Топ-15 регионов ===')
profile_df.filter('addrref_region_uk IS NOT NULL') \
    .groupBy('addrref_region_uk').count() \
    .orderBy('count', ascending=False).show(15, truncate=False)

print('=== Модельный сегмент ===')
profile_df.groupBy('srvpackagesegment_ccode').count().orderBy('count', ascending=False).show(20, truncate=False)

print('=== Покрытие полей (% NOT NULL) ===')
from pyspark.sql import functions as F
total = profile_df.count()
for col in ['clientcounterparty_flag', 'sparkokved_ccode', 'addrref_region_uk',
            'addrref_city_name', 'srvpackagesegment_ccode', 'registry_authcapital_amt']:
    if col in profile_df.columns:
        non_null = profile_df.filter(F.col(col).isNotNull()).count()
        print(f'  {col:40s} {non_null:>10,} / {total:,}  ({100*non_null/total:.1f}%)')

In [ ]:
profile_df.show(10, truncate=30, vertical=True)

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **clientcounterparty_flag** | Y = клиент банка, N = контрагент (проспект), ? = неизвестно |
| **clientchange_date** | Дата перехода из контрагента в клиенты |
| **sparkokved_ccode** | Код ОКВЭД из системы СПАРК (для ЮЛ) |
| **addrref_region_uk** | ID региона из справочника адресов |
| **srvpackagesegment_ccode** | Код сегмента по уровню обслуживания (модельный сегмент) |
| **registry_authcapital_amt** | Зарегистрированный уставный капитал |
| **revenue_rur_amt** | Годовая выручка из СПАРК (если доступна) |
| **SCD (shist)** | Slowly Changing Dimension — таблица с историей изменений (effective_from/to) |

---

**Следующий шаг**: `02_etl_behavior.ipynb` — поведенческие агрегаты.